In [ ]:
spark.stop()

In [ ]:
# наименование задачи/заказчика
name = ''

In [ ]:
User_prefix = 'dvo_none'
User_postfix = 'b2b'
Jira_task = f'{name}'

In [ ]:
# инн заказчика для исключения текущих клиентов (в виде списка)
inn_client = [
    ''
]

In [ ]:
import sys
import os
import warnings
from IPython.display import display, HTML, clear_output
# display(HTML("<style>.container {width:80% !important;}</style>"))
warnings.filterwarnings("ignore")
sys.path.append('..')
sys.path.append('../../../../../')
sys.path.append('../../../../')
sys.path.append('../../../')
sys.path.append('../../')
sys.path.append('../../../../../box/')
sys.path.append("../../../../.local/lib/python3.6/site-packages")
import sh
from datasol_utils_py.base_utils import read_config, get_spark_session, DataSolPath

In [ ]:
# PYTHON_PYSPARK_PATH = "/opt/sdp/mlpy3811v23/bin/python"
# PYTHON_PYSPARK_PATH = "/opt/sdp/py38zno20008661/bin/python"
user_config = read_config('../../configs/user_config.yaml')
spark_config = read_config('../../configs/spark_config.yaml')['sdp_spark3']
spark_config['setMaster'] = 'yarn'
spark_config['spark.executor.instances'] = 5
spark_config['spark.executor.cores'] = 15
spark_config["spark.driver.maxResultSize"] = "64G"
spark_config['spark.yarn.access.hadoopFileSystems'] = 'hdfs://nsld3:8020/,hdfs://arnsdpsbx:8020/'
spark_config['spark.sql.shuffle.partitions'] = 500
spark_config['hive.exec.dynamic.partition.mode'] = 'nonstrict'
spark_config['spark.sql.autoBroadcastJoinThreshold'] = -1
spark_config['spark_sdp_path_1'] = '/usr/sdp/current/spark3-client/python/'
spark_config['spark_sdp_path_2'] = '/usr/sdp/current/spark3-client/python/lib/py4j-0.10.9.3-src.zip'
spark_config['SPARK_HOME'] = '/usr/sdp/current/spark3-client/'
# spark_config['spark.yarn.queue'] = "default"
spark_config['spark.yarn.queue'] = "datascience"
spark_config['PYSPARK_PYTHON'] = sys.executable
spark_config['PYSPARK_DRIVER_PYTHON'] = sys.executable
# spark_config['PYSPARK_PYTHON'] = '/opt/cloudera/parcels/PYENV.AUTOML-3.6.pyenv.p0.3/bin/python3'
# spark_config['PYSPARK_DRIVER_PYTHON'] = '/opt/cloudera/parcels/PYENV.AUTOML-3.6.pyenv.p0.3/bin/python3'
spark_config
user_config = read_config('../../configs/user_config.yaml')
config = read_config('../../configs/user_config.yaml')

In [ ]:
spark = get_spark_session(spark_config, application_name='Drokin_VO_B2B')
print("Контекст запущен")

In [ ]:
do_sql = lambda x: spark.sql(x)

# функция удаления таблиц 
drop_table = lambda x: do_sql("drop table if exists {}".format(x))

# функция для создания таблиц по названию и SQL запросу
show_head = lambda x: do_sql(f"select * from {x} limit 10").toPandas()

# функция для проверки размера таблицы
def get_size(table):
    buf = do_sql(f"describe extended {table}").toPandas()
    path = buf[buf['col_name']=='Location']['data_type'].iloc[0]
    l = str(sh.hdfs("dfs","-count",path)).split(" ")
    l = [x for x in l if len(x) > 0]
    part = int(l[1])
    size = int(l[2])
    if part > 1:
        size_per_part = size/(part - 1)
    else:
        size_per_part = size
    return {
        "size":size,
        "partitions":part,
        "size_per_partition":round(size_per_part),
        'path':path
    }

# Функция, чтобы не писать каждый раз spark.sql([текст запроса spark-sql]),
# а писать вместо этого просто sql([текст запроса spark-sql])
sql = lambda x : spark.sql(x)

# можно писать show([текст запроса spark-sql]) для вывода результатов какого-то запроса к таблицам
show = lambda x : display(spark.sql(x).toPandas())

# Функция создания новой таблицы из запроса
def create_new_table_as(table_nm, query):
    _ = sql(f'drop table if exists {table_nm}')
    _ = sql(f"""
        CREATE TABLE {table_nm}
            STORED AS ORC
            TBLPROPERTIES ('orc.compress'='SNAPPY')
        AS
        {query}
    """)
    
block_start = lambda x : print("\nSTART: {0} at {1}".format(x,datetime.now()))
block_end = lambda x : print("\nEND: {0} at {1}".format(x,datetime.now()))

In [ ]:
# Любые необходимые пакеты
import numpy as np
import pandas as pd
from IPython.display import display
import time
from pyspark.sql import window as w
from datetime import datetime, timedelta
import itertools as it
from tqdm import tqdm
# import box_utils

import warnings
warnings.filterwarnings('ignore')

# Любые необходимые пакеты
import numpy as np
import pandas as pd
import random as rand
import string
import os
import joblib
import re
import calendar
import time
from IPython.display import display
from datetime import datetime, timedelta
from pandas.tseries.offsets import MonthEnd
import itertools as it
from collections import Counter, defaultdict
from functools import reduce
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

In [ ]:
import pyspark
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import isnan, isnull
from pyspark.sql import types as t
from pyspark.sql import Window
from pyspark.sql.functions import regexp_replace
sys.path.append('../../../box')


pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 200)

In [ ]:
CLUSTER_NAME = user_config['save_scheme']['cluster_name']
TEAM_NAME = user_config['save_scheme']['team_name']
PROJECT_HIVE_FOLDER = user_config['save_scheme']['project_hive_folder']
USER_PREFIX = user_config['suffixies']['user_name']
POSTFIX = user_config['suffixies']['postfix']
JIRA_TASK = user_config['suffixies']['jira_task']
scheme = CLUSTER_NAME + "_" + TEAM_NAME + "_" + PROJECT_HIVE_FOLDER
prefix = USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX + "_"

saver = DataSolPath(cluster_name=CLUSTER_NAME, 
                    team_name=TEAM_NAME, 
                    project_hive_folder=PROJECT_HIVE_FOLDER)

# Витрины источники
census_ext_tbl_nm =    "arnsdpsbx_team_monetization.census_ext_sv"
trn_link_tbl_nm =      "arnsdpsbx_team_monetization.trn_link_sv" #парные транзакции
mt_init_tbl_nm =       "arnsdpsbx_team_monetization.mt_init"
twogis_table_nm =      "prx_gis_data_custom_cib_p4d_2gis.branch"
trc_categories_nm =    "arnsdpsbx_team_monetization_prt_adhoc.trc_categories" #словарь категорий по мсс
foreign_cards_tbl_nm = "arnsdpsbx_team_monetization_prt_adhoc.mmt_foreign_cards"
base_fl_income =       "arnsdpsbx_t_team_monetization_opfl.base_fl_income"
basis_client =         "prx_aaa_products_custom_cib_products.basis_client"
sbersov_path =   "prx_aaa_products_custom_cib_products.basis_transactions"
base_inn_segment_tb_gosb_km_tkm = "arnsdpsbx_team_monetization_prt_adhoc.base_inn_segment_tb_gosb_km_tkm"
basis_client_hist = "prx_aaa_products_custom_cib_products.basis_client_hist"


#Создаваемые в проекте таблицы

scheme = f'{scheme}.{prefix}'

array_of_dates_tbl_nm =     scheme + 'array_of_dates'
trc_geocoded_tbl_nm =       scheme + 'trc_geocoded'
trc_beacons_tbl_nm =        scheme + 'beacons'
pair_trans_beacons_candidates_tbl_nm = scheme + 'pair_trans_beacons_candidates'
trc_markup_tbl_nm =         scheme + 'markup'
terminals_tbl_nm =          scheme + 'terminals'
transaction_tbl_nm =        scheme + 'transactions'
features_wo_tbl_nm =        scheme + 'features'
prediction_tbl_nm =         scheme + 'preds'
razmetka_tbl_nm =           scheme + 'razmetka'
razmetka_tbl_enriched_nm =  scheme + 'razmetka_enriched'
razmetka_brands_tbl =       scheme + 'razmetka_brands_tbl'
terminals_in_tbl_nm =       scheme + 'terminals_in_tbl'
transactions_in_tbl_nm =    scheme + 'transactions_in_tbl'
trc_clients_tbl_nm =        scheme + 'trc_clients_tbl'
transactions_out_tbl_nm =   scheme + 'transactions_out_tbl'
rfm_aggregate_tbl_nm =      scheme + 'rfm_aggregate_tbl'
clients_age_gender_tbl_nm = scheme + 'clients_age_gender_tbl'
interests_tbl_nm =          scheme + 'interests_tbl'
clients_income_tbl_nm =     scheme + 'clients_income_tbl'
clients_profile_tbl_nm =    scheme + 'clients_profile_tbl'
trans_in_and_profile_tbl =  scheme + 'trans_in_and_profile'
trans_out_and_profile_tbl = scheme + 'trans_out_and_profile_tbl'
trc_params_tbl_nm =         scheme + 'trc_params_tbl'
trc_parameters_tbl_nm =     scheme + 'trc_parameters_tbl'
soc_dem_aggregate_tbl_nm =  scheme + 'soc_dem_aggregate_tbl'
top_categories_in_tbl_nm =  scheme + 'top_categories_in_tbl'
top_categories_out_tbl_nm = scheme + 'top_categories_out_tbl'
top_brands_in_tbl_nm =      scheme + 'top_brands_in_tbl'
razmetka_top_brands_out_tbl_nm = scheme + 'razmetka_top_brands_out_tbl'
trans_out_brands_and_profile_tbl_nm = scheme + 'trans_out_brands_and_profile_tbl'
top_brands_out_tbl_nm =     scheme + 'top_brands_out_tbl'

distribution_chains_tbl_nm =scheme + 'distribution_chains_tbl'
related_brands_tbl_nm =     scheme + 'related_brands_tbl'
client_transactions = scheme + 'client_transaction'
epk_fot_tbl = scheme + 'epk_fot'
epk_amount_trn_transactions_tbl = scheme + 'epk_amount_trn_transactions'
epk_amount_trn_transactions_tbl2 = scheme + 'epk_amount_trn_transactions2'
epk_amount_trn_transactions_reserve_tbl = scheme + 'epk_amount_trn_transactions_reserve'
census_merchant_id_inn_tbl = scheme + 'census_merchant_id_inn'
inn_clients_okved_tbl = scheme + 'census_merchant_id_inn'
pharmacy_medical_center_info_tbl = scheme + 'inn_clients_okved'
inn_clients_competitors_tbl = scheme + 'inn_clients_competitors'
inn_clients_competitors_tbl2 = scheme + 'inn_clients_competitors2'
inn_client_competitors_joining_tbl = scheme + 'inn_client_competitors_joining'
already_clients_tbl = scheme + 'already_clients'
already_clients_1_tbl = scheme + 'already_clients_1'
#Побочные таблицы
brands_markup_collect_tbl = "arnsdpsbx_team_monetization_prt_adhoc.brands_markup_collection"

In [ ]:
ml360_folder = 'prx_bazovaya_podpiska_ml360_custom_cib_ml360.{}'
credit_products_folder = 'prx_credit_products_ml360_custom_cib_ml360_credit_products.{}'

In [ ]:
do_sql("""
Refresh table prx_ml360_credit_products_b2b_custom_cib_ml360_credit_products.u_transaction_classification""").toPandas()

In [ ]:
sbersov_path = 'prx_ml360_credit_products_b2b_custom_cib_ml360_credit_products.u_transaction_classification'
sbersov_table = spark.read.table(sbersov_path)

In [ ]:
def clear_punct(x):
    return re.sub(' +', ' ', re.sub('ё','е', re.sub(r'[^\w\s\-\/,]', ' ', x.lower())))
clear_punct_udf = F.udf(clear_punct, StringType())

# Поиск по словам СберСоветника

In [ ]:
words_finadv = sbersov_table
words_finadv = words_finadv[
    (~words_finadv['c_nazn'].isNull())&(words_finadv['short_dt'] >= '2025-01-01')\
    &(words_finadv['short_dt'] <= '2026-01-01')
]

In [ ]:
# создаем датафрейм из слов сберсоветника для поиска
lst_sbersov = []

df_sbersov_spark = spark.createDataFrame(pd.DataFrame(lst_sbersov, columns = ['word']))

In [ ]:
df_insbersov = words_finadv.join(df_sbersov_spark, on = 'word', how = 'inner')

# Регулярки

In [ ]:
list_words = []





In [ ]:
re.findall(list_words[-1], '')

In [ ]:
words_finadv_upd = sbersov_table
words_finadv_upd_ = words_finadv_upd[
    (~words_finadv_upd['c_nazn'].isNull())&(words_finadv_upd['short_dt'] >= '2025-01-01')\
    &(words_finadv_upd['short_dt'] <= '2026-01-01')
]

In [ ]:
words_finadv_all = words_finadv_upd_.withColumn('c_nazn', F.concat(clear_punct_udf('c_nazn'), F.lit(' ')))
words_finadv_all = words_finadv_all[~words_finadv_all['c_nazn'].isNull()]

In [ ]:
# Собираем активных клиентов заказчика
inn_cl = spark.createDataFrame(pd.DataFrame(inn_client, columns = ['inn']))

already_clients = words_finadv_upd.join(F.broadcast(inn_cl), words_finadv_upd.inn_kt == inn_cl.inn, how = 'leftsemi')\
                                  .withColumnRenamed('inn_dt', 'inn')\
                                  .select('inn').union(inn_cl).distinct()
        
already_clients_1 = words_finadv_upd.join(F.broadcast(inn_cl), words_finadv_upd.inn_dt == inn_cl.inn, how = 'leftsemi')\
                                    .withColumnRenamed('inn_kt', 'inn')\
                                    .select('inn').union(inn_cl).distinct()

In [ ]:
tpm_plat2 = "arnsdpsbx_team_monetization_prt_adhoc.USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX + "_" + ALREADY_CLIENTS"

In [ ]:
(already_clients
.repartition(1)
.write
.mode('overwrite').format('parquet')
.option('compression','snappy')
 #.repartition(1)
 #.partiotionBy('month')
.saveAsTable(tpm_plat2))

In [ ]:
already_clients = spark.table(tpm_plat2)

In [ ]:
already_clients.count()

In [ ]:
tpm_plat3 = "arnsdpsbx_team_monetization_prt_adhoc.USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX + "_" + ALREADY_CLIENTS_1"

In [ ]:
(already_clients_1
 .repartition(1)
.write
.mode('overwrite').format('parquet')
.option('compression','snappy')
 #.repartition(1)
 #.partiotionBy('month')
.saveAsTable(tpm_plat3))

In [ ]:
already_clients_1 = spark.table("arnsdpsbx_team_monetization_prt_adhoc.USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX + "_" + ALREADY_CLIENTS_1")

In [ ]:
already_clients_1.count()

In [ ]:
# Функция для поиска слов в назначении
search_keyw_plat = F.udf(lambda x: ', '.join(set(re.findall(r'|'.join(list_words), x))), StringType()) 
# search_keyw_pol = F.udf(lambda x: ', '.join(set(re.findall(r'|'.join(list_sell), x))), StringType())

words_finadv_plat = words_finadv_all.withColumn('search_keyw', search_keyw_plat(words_finadv_all['c_nazn']))
# words_finadv_pol = words_finadv_all.withColumn('search_keyw', search_keyw_pol(words_finadv_all['c_nazn']))

In [ ]:
# Оставляем только те платежки, где найдены необходимые слова 
words_finadv_plat = words_finadv_plat.filter(F.col('search_keyw') != '')
# words_finadv_pol = words_finadv_pol.filter(F.col('search_keyw') != '')

# Обработка найденных платежек

In [ ]:
# Создаем датафрейм из id транзакции и слова, найденного в этой транзакции, и по регуляркам, и по сберсоветнику. 
# Джойним их, получая наиболее полный список id транзакций.
# Полученный список джойним со всей витриной платежек
trans_id = words_finadv_plat[['id','search_keyw']].groupby(['id']).agg(F.collect_set('search_keyw').alias('re_word')).join(
    df_insbersov[['id','word']].groupby(['id']).agg(F.collect_set('word').alias('fa_word')),
    on=['id'], how='outer').fillna('')

words_selected_plat = words_finadv_all.join(trans_id, on=['id'], how='inner')
            
# words_selected_pol = words_finadv_all.join(trans_id, on=['id'], how='inner')

In [ ]:
tpm_plat6 = "arnsdpsbx_team_monetization_prt_adhoc.USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX"

In [ ]:
(words_selected_plat
 .repartition(1)
.write
.mode('overwrite').format('parquet')
.option('compression','snappy')
 #.repartition(1)
 #.partiotionBy('month')
.saveAsTable(tpm_plat6))

In [ ]:
words_selected_plat = spark.table(tpm_plat6)

In [ ]:
words_selected_plat.count()

In [ ]:
words_selected_plat.show(100, False, True) #обязательная проверка на мусорные транзакции

In [ ]:
words_selected_plat_grouped = words_selected_plat.withColumnRenamed('inn_dt','inn').groupby(['inn']).agg(
    F.sum('c_sum').cast(FloatType()).alias('trans_sum'), 
    F.countDistinct('id').alias('trans_cnt'),
    F.collect_set('re_word').alias('re_word'),
    F.collect_set('word').alias('fa_word')
).withColumn('type', F.lit('plat')).filter(F.length(F.col('inn')) < 12)

# words_selected_pol_grouped = words_selected_pol.withColumnRenamed('inn_kt','inn').groupby(['inn']).agg(
#     F.sum('c_sum').cast(FloatType()).alias('trans_sum'), 
#     F.countDistinct('id').alias('trans_cnt'),
#     F.collect_set('search_keyw').alias('re_word'), 
#     F.collect_set('word').alias('fa_word') 
# ).withColumn('type', F.lit('pol'))
# search_keyw  word

In [ ]:
all_inn = words_selected_plat_grouped.count()
all_inn

# Фильтрация найденных платежек

In [ ]:
# Фильтрация по сегменту и в хождению в один холдинг с крупнейшими предприятиями
ok_segment_lst = ['Микро', 'Малые', 'Средние', 'Крупные']

crm = spark.read.table(ml360_folder.format('u_sparkinterfax_crm'))\
               .select('inn', 'crm_segment', 'crm_holding_id')
    
krup_hold = crm.filter(F.col('crm_segment') == 'Крупнейшие').select('crm_holding_id')

marketplaces = ['7802754982', '7721546864', '7714326836',
                '7704217370', '7704357909', '9701048328',
                '9709073460', '7705935687', '7703380158',
                '6194002171']

marketplaces_df = spark.createDataFrame(pd.DataFrame(marketplaces, columns = ['inn']))

crm_filtered = crm.join(F.broadcast(krup_hold), on = 'crm_holding_id', how = 'leftanti')\
                  .filter(F.col('crm_segment').isin(ok_segment_lst))\
                  .join(F.broadcast(marketplaces_df), on = 'inn', how = 'leftanti')

In [ ]:
lplat_filtered_1 = words_selected_plat_grouped.join(crm_filtered, on = 'inn', how = 'inner')

# lpol_filtered_1 = words_selected_pol_grouped.join(crm_filtered, on = 'inn', how = 'inner')

In [ ]:
lplat_filtered_1.count()

In [ ]:
# Фильтрация по ОКФС и флагу активности
okfs = spark.read.table(ml360_folder.format('u_sparkinterfax_integrum'))\
                .select('inn', 'okfs_cd', 'active_flag', 'status_nm')\
                .filter(F.col('active_flag')==1)\
                .withColumn('okfs_cd', F.col('okfs_cd').cast(DoubleType()))\
                .filter((F.col('okfs_cd') < 20) | (F.col('okfs_cd') >= 40)).fillna({'status_nm':''})\
                .filter(~F.lower(F.col('status_nm'))
                       .rlike(r'(?:ликвидац|банкрот|прекратит деятельность|предстоящем исключении|регистрация признана ошибочной|признано ошибочным)'))

In [ ]:
lplat_filtered_2 = lplat_filtered_1.join(okfs, on = 'inn', how = 'inner')

# lpol_filtered_2 = lpol_filtered_1.join(okfs, on = 'inn', how = 'inner')

In [ ]:
lplat_filtered_2.count()

# Фильтр по РЕГИОНАМ

In [ ]:
txt = '''

'''

inp_regs = [i.strip() for i in re.sub(';', '', re.sub(r' +', ' ', txt)).split('\n')]
subjets = spark.read.table('arnsdpsbx_t_team_monetization_products.ens_dict_cc_region_okato').select('region').distinct().collect()
list_subj = [i[0] for i in subjets]

mb_subj = []
a,b,c = 0,0,0
for i in inp_regs[1:-1]:
    te = process.extract(i, list_subj, limit = 3)
    if te[0][1] >= 90:
        mb_subj.append(te[0][0])
        a+=1
    elif (te[0][1] >= 40)&(te[0][1] < 90):
        print(i, te, end = '\n\n')
        mb_subj.append(te[0][0])
        b+=1
    else:
        print('ошибка',i, te, end = '\n\n')
        c+=1
        
print(f'Точно - {a}, Проверить - {b}, Не нашлось - {c}')
mb_subj

In [ ]:
# Распределение по регионам
integrum = spark.read.table(ml360_folder.format('u_sparkinterfax_integrum'))\
                    .select('inn', 'okato_cd')\
                    .withColumn('okato', F.col('okato_cd')[0:2])
        
# okato = spark.read.parquet('okato').withColumnRenamed('okato_cd', 'okato')  

inn_wth_regions = integrum.join(F.broadcast(spark.table("arnsdpsbx_t_team_monetization_products.ens_dict_cc_region_okato")), on = 'okato', how = 'inner')


regions = []

if len(regions) > 0:
    inn_wth_regions = inn_wth_regions.filter(F.col('region').isin(regions))

In [ ]:
lplat_filtered_3 = lplat_filtered_2.join(inn_wth_regions, on = 'inn', how = 'inner')

In [ ]:
after_region = lplat_filtered_3.count()
after_region

# Фильтр по ОКВЭД

In [ ]:
okved_part = spark.read.table('prx_bazovaya_podpiska_ml360_custom_cib_ml360.u_all_okved_hist').filter(F.col('mon')=='2024-03-31')

In [ ]:
# hive.read.table(ml360_folder.format('u_all_okved_hist'))\
okved_list = []


okved_ = okved_part\
                 .filter(F.col('okved_original_version') == 2)\
                 .drop_duplicates(subset=['inn'])\
                 .select('inn', 'okved')\
#  .filter(F.col('main_flag') == 1)

# if len(okved_list) > 0:
#     okved = okved_.filter(F.col('main_flag') == 1)\
#                   .filter(F.col('okved').isin(okved_list))\
#                   .drop_duplicates(subset=['inn'])\
#                   .select('inn', 'okved')
# else:
#     okved = okved_.filter(F.col('main_flag') == 1)\
#                   .drop_duplicates(subset=['inn'])\
#                   .select('inn', 'okved')
        

In [ ]:
lplat_filtered_4 = lplat_filtered_3.join(okved_, on = 'inn', how = 'inner')

# lpol_filtered_4 = lpol_filtered_3.join(okved, on = 'inn', how = 'inner')

In [ ]:
lplat_filtered_4.count()

# Исключение АКТИВНЫХ

In [ ]:
# Минус активные клиенты заказчика
lplat_filtered_5 = lplat_filtered_4.join(F.broadcast(already_clients), on = 'inn', how = 'leftanti')\
                                   .join(F.broadcast(already_clients_1), on = 'inn', how = 'leftanti')

# lpol_filtered_5 = lpol_filtered_4.join(F.broadcast(already_clients), on = 'inn', how = 'leftanti')\
#                                  .join(F.broadcast(already_clients_1), on = 'inn', how = 'leftanti')

In [ ]:
after_active = lplat_filtered_5.count()
after_active

# Исключение из списка из xlsx или списка в массиве

In [ ]:
df = pd.DataFrame(data=[], columns=['inn'])
df

In [ ]:
inn_clients_iskl = spark.createDataFrame(df)

In [ ]:
excl_clietns = lplat_filtered_5.join(inn_clients_iskl, on = 'inn', how = 'leftanti')

In [ ]:
after_inn_clients = excl_clietns.count()
after_inn_clients

# Фильтр по ВЫРУЧКЕ

In [ ]:
# Выручка
revenue = spark.read.table(ml360_folder.format('yr_fin_statement'))\
                   .withColumn('revenue', F.col('fin_stmt_2110_amt')*1000).select('inn', 'revenue', 'yr')

partition = Window.partitionBy('inn').orderBy(F.col('yr').desc())

revenue_2022_2023 = revenue.withColumn('rn', F.row_number().over(partition))\
                     .select('inn', 'revenue', 'yr', 'rn')\
                     .filter(F.col('rn') == 1)\
                     .filter(F.col('yr')[0:4] > '2022')\
                     .withColumnRenamed('yr', 'year_revenue')\
                     .drop('rn')

start_rev = 100_000_000
end_rev = 222_500_000_000
revenue_filtered = revenue_2022_2023.filter(F.col('revenue') >= start_rev)

In [ ]:
lplat_filtered_6 = lplat_filtered_5.join(revenue_filtered, on = 'inn', how = 'inner')
# excl_clietns
# lpol_filtered_6 = lpol_filtered_5.join(revenue_filtered, on = 'inn', how = 'inner')

In [ ]:
after_revenue = lplat_filtered_6.count()
after_revenue

In [ ]:
after_revenue_1 = lplat_filtered_6_1.count()
after_revenue_1

In [ ]:
# Функция для извлечения общей суммы
get_sum = lambda x: int(x.select(F.sum('trans_sum')).collect()[0][0])

In [ ]:
get_sum(lplat_filtered_6)

# Материальные критерии

In [ ]:
# Фильтрация по объему/количеству закупок/продаж
trans_sum_ust_down = 10_000_000
trans_cnt_ust_down = 3

lplat_filtered_7 = lplat_filtered_6.filter((lplat_filtered_6['trans_sum'] >= trans_sum_ust_down) & 
                                           (lplat_filtered_6['trans_cnt'] >= trans_cnt_ust_down))

# lpol_filtered_7 = lpol_filtered_6.filter((lpol_filtered_6['trans_sum'] >= trans_sum_ust_down) & 
#                                          (lpol_filtered_6['trans_cnt'] >= trans_cnt_ust_down))

In [ ]:
lplat_filtered_7.count()

In [ ]:
lplat_filtered_7.select(F.col("inn")).toPandas()

In [ ]:
get_sum(lplat_filtered_7)

In [ ]:
get_sum(lplat_filtered_7_1)

In [ ]:
lplat_filtered_7\
        .toPandas()\
        .to_excel(f'{name}_calc.xlsx', index = False)

In [ ]:
prx_cib_ml360_arnsdp_custom_cib_ml360.

# Подготовка отчета

In [ ]:
tpm_plat10 = "arnsdpsbx_team_monetization_prt_adhoc.arnsdpsbx_team_monetization_prt_adhoc.USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX + "_" + PREFIN"

In [ ]:
(lplat_filtered_7
 .repartition(1)
.write
.mode('overwrite').format('parquet')
.option('compression','snappy')
 #.repartition(1)
 #.partiotionBy('month')
.saveAsTable(tpm_plat10))

In [ ]:
lplat_filtered_7 = spark.table("arnsdpsbx_team_monetization_prt_adhoc.arnsdpsbx_team_monetization_prt_adhoc.USER_PREFIX + "_" + JIRA_TASK + "_" + POSTFIX + "_" + PREFIN")

In [ ]:
def generate_column(lst):
    borders = []
    for i in range(5):
        part = lst[int(len(lst)/5*i):int(len(lst)/5*(i+1))]
        borders.append(min(part))
    borders.append(max(lst))
    
    return borders

def count_borders(table):
    sum_lst = sorted([int(i[0]) for i in table.select('trans_sum').collect()])
    cnt_lst = sorted([int(i[0]) for i in table.select('trans_cnt').collect()])
    
    borders_sum = generate_column(sum_lst)
    borders_cnt = generate_column(cnt_lst)
    
    column = lambda x:['({} - {}]'.format(x[i], x[i+1]) for i in range(len(x)-1)]
   
    borders_df = pd.DataFrame(columns = ['Рейтинг объема', 'Диапазон объема, руб', 'Рейтинг частоты', 'Диапазон частоты, раз'])
    borders_df['Рейтинг объема'] = borders_df['Рейтинг частоты'] = [1,2,3,4,5]
    borders_df['Диапазон объема, руб'] = column(borders_sum)
    borders_df['Диапазон частоты, раз'] = column(borders_cnt)
    
    return borders_df, borders_sum, borders_cnt

In [ ]:
# Всегда делать!
borders_df_plat, borders_sum_plat, borders_cnt_plat = count_borders(lplat_filtered_7)
borders_df_plat.to_excel('.xlsx', index = False)

In [ ]:
lids_table_rank_plat = lplat_filtered_7.withColumn('sum_group', F.when(F.col('trans_sum') <= borders_sum_plat[1], 1)
                                                      .when(F.col('trans_sum') <= borders_sum_plat[2], 2)
                                                      .when(F.col('trans_sum') <= borders_sum_plat[3], 3)
                                                      .when(F.col('trans_sum') <= borders_sum_plat[4], 4)
                                                      .otherwise(5))

lids_table_rank_plat = lids_table_rank_plat.withColumn('cnt_group', F.when(F.col('trans_cnt') <= borders_cnt_plat[1], 1)
                                                      .when(F.col('trans_cnt') <= borders_cnt_plat[2], 2)
                                                      .when(F.col('trans_cnt') <= borders_cnt_plat[3], 3)
                                                      .when(F.col('trans_cnt') <= borders_cnt_plat[4], 4)
                                                      .otherwise(5))

In [ ]:
sum_desc_plat = lplat_filtered_7.select('trans_sum', 'trans_cnt').summary('mean', 'stddev')

sum_desc_p_plat = sum_desc_plat.toPandas()
sum_desc_p_plat.index = sum_desc_p_plat.summary

sum_desc_p_plat['trans_sum'] = pd.to_numeric(sum_desc_p_plat['trans_sum'], downcast='float')
sum_desc_p_plat['trans_cnt'] = pd.to_numeric(sum_desc_p_plat['trans_cnt'], downcast='float')

mean_sum_plat = float(sum_desc_p_plat['trans_sum']['mean'])
mean_cnt_plat = float(sum_desc_p_plat['trans_cnt']['mean'])
stddev_sum_plat = float(sum_desc_p_plat['trans_sum']['stddev'])
stddev_cnt_plat = float(sum_desc_p_plat['trans_cnt']['stddev'])

In [ ]:
lids_table_rank_score_plat = lids_table_rank_plat.withColumn('z_score', 
F.when(((F.col('trans_sum') - mean_sum_plat) / stddev_sum_plat + (F.col('trans_cnt') - mean_cnt_plat) / stddev_cnt_plat) / 2  * 2/3 + 3> 5, 5)
 .when(((F.col('trans_sum') - mean_sum_plat) / stddev_sum_plat + (F.col('trans_cnt') - mean_cnt_plat) / stddev_cnt_plat) / 2  * 2/3 + 3< 1, 1)
 .otherwise(((F.col('trans_sum') - mean_sum_plat) / stddev_sum_plat + (F.col('trans_cnt') - mean_cnt_plat) / stddev_cnt_plat) / 2 * 2/3 + 3))

# del lids_table_rank_plat
# lids_table_rank_score.show()

In [ ]:
lids_table_rank_score_plat.toPandas()

In [ ]:
okved_dict_v2 = spark.read.table('arnsdpsbx_team_monetization_prt_adhoc.okved_ref')\
                         .fillna({'OKVED_NM': ''})\
                         .filter(F.col('VERSION') == 2)\
                         .select('OKVED_CD', 'OKVED_NM')\
                         .withColumnRenamed('OKVED_CD', 'okved')

okved_ = okved_part\
                 .filter(F.col('main_flag') == 1)\
                 .drop_duplicates(subset=['inn'])\
                 .select('okved')

okved_2 = okved_.join(okved_dict_v2, 'okved', 'inner').distinct()

In [ ]:
lids_table_rank_score_plat_2 = lids_table_rank_score_plat.join(okved_2, 'okved', 'inner')

In [ ]:
triema_df = spark.table("arnsdpbsx_team_monetization_prt_adhoc.triema_final_FINAL")

t2 = spark.table("px_aaa_products_custom_clb_products_basis_client")

pt = (
    lids_table_rank_score_plat_2.alias("t1")
    .join(
        t2.alias("t2"),
        (F.col("t1.inn") == F.col("t2.inn")) &
        (F.col("t2.is_inactive_fns") == 0),
        "inner"
    )
    .select(
        F.col("t1.inn").alias("inn"),
        F.col("t2.short_name"),
        F.col("t2.okpo"),
        F.col("t2.ogrn")
    )
    .distinct()
)

withname_tbl4 = (
    lids_table_rank_score_plat_2.alias("mt")
    .join(
        pt.alias("pt"),
        F.col("mt.inn") == F.col("pt.inn"),
        "inner"
    )
    .select(
        F.col("mt.*"),
        F.col("pt.short_name"),
        F.col("pt.okpo"),
        F.col("pt.ogrn")
    )
)


In [ ]:
withname_tbl4 = do_sql(f"""
Select mt.*,
pt.short_name,
pt.okpo,
pt.ogrn
From arnsdpsbx_team_monetization_prt_adhoc.triema_final_FINAL mt
INNER JOIN (Select distinct
t1.inn,
t2.short_name,
t2.okpo,
t2.ogrn
From arnsdpsbx_team_monetization_prt_adhoc.triema_final_FINAL t1
INNER JOIN prx_aaa_products_custom_cib_products.basis_client t2
ON t1.inn = t2.inn and t2.is_inactive_fns = 0) pt
ON mt.inn = pt.inn""")

In [ ]:
withname_tbl4.limit(2).toPandas()

In [ ]:
columns = [
'inn',
'short_name',
'trans_sum',
'okved',
'OKVED_NM',
'okpo',
'ogrn',
'region',
'revenue',
'sum_group',
'cnt_group',
'z_score',
'fa_word'


]
# 'type',

In [ ]:
report = withname_tbl4.select(columns)

In [ ]:
report_pan = report.toPandas()

In [ ]:
d = {'inn': 'ИНН', 
     'okved': 'ОКВЭД', 
     'trans_sum': 'Объем потребления', 
     'trans_cnt': 'Количество покупок', 
     're_word': 'Потребляемая продукция', 
     'fa_word': 'Потребляемая продукция(разметка)', 
     'type': 'Тип покупатель', 
     'sum_group': 'Рейтинг объема покупок', 
     'cnt_group': 'Рейтинг частоты покупок', 
     'z_score': 'Скор-балл покупка', 
     'short_name': 'Наименование',
     'region': 'Регион', 
     'OKVED_NM': 'Наименование ОКВЭД', 
     'okpo': 'ОКПО', 
     'ogrn': 'ОГРН',
     'okfs_cd': 'ОКФС',
     'sum_group_pol': 'Рейтинг объема поставок',
     'cnt_group_pol': 'Рейтинг частоты поставок',
     'z_score_pol': 'Скор-балл поставка',
     're_word_pol': 'Поставляемая продукция',
     'type_pol': 'Тип поставщик',
     'revenue': 'Выручка'
     
}
dicted = []
for i in report_pan.columns:
    if d.get(i) != None:
        dicted.append(d.get(i))
    else:
        dicted.append(i)

In [ ]:
dicted

In [ ]:
report_pan.columns = dicted

In [ ]:
report_pan

In [ ]:
report_pan.to_excel('report_' + name + '.xlsx', index = False)

# Коммерческий интерес 

In [ ]:
import pandas as pd
from IPython.core.display import display, HTML
display(HTML("<style>.container {width:80% !important;}</style>"))

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

pd.set_option('display.max_rows', 70)

from sklearn.metrics import precision_recall_curve
import json
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta
from sklearn.calibration import IsotonicRegression
from sklearn.calibration import calibration_curve
import matplotlib.lines as mlines
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

import pandas as pd
import re
import numpy as np
import string
import os
import joblib
from datetime import datetime as dt
from dateutil.relativedelta import relativedelta
from fuzzywuzzy import fuzz
from fuzzywuzzy import process
import time
from pyspark.sql.window import Window
import sys
import warnings

In [ ]:
#сюда указывать отобранные слова
words_list = [
'Датчик уровня',
'Датчик давления масла',
'Датчик давления',
'Датчик скорости',
'Датчик температуры',
'Термопара',
'Манометр',
'Термометр',
'Счетчик воды',
'Счетчик газа',
'Газоанализатор',
'Анализатор'
]

In [ ]:
#function for getting the last day of month

def last_day_of_month(any_day):
    next_month = any_day.replace(day=28) + datetime.timedelta(days=4)
    return next_month - datetime.timedelta(days=next_month.day)

In [ ]:
def precision_recall(df, prob_thr = '300', group_by = 'max', last_date_analysis = '2021-05-31'):
    
    prob_col = 'prob_{}'.format(prob_thr)
    target_col = 'target{}_new'.format(prob_thr)
    
    df_avg= df.groupby(['inn','control_date'], as_index = False)\
                  .agg({prob_col: group_by, target_col: 'max'}) 
    

    df_anl2 = df_avg[df_avg['control_date'] == last_date_analysis] 
    
    
    prs, rcl, thr = precision_recall_curve(df_anl2[target_col], df_anl2[prob_col])

    
    return df_avg, df_anl2, prs, rcl, thr

In [ ]:
#prob_thr - порог для покупки/продажи: 100, 300, 500, 1m или 2m (соответственно 100 тысяч, ... , 2 миллиона)

def create_sample(df, words_list, precision_thr = 0.4, 
                 last_date_analysis = '2021-05-31', sample_date = '2021-08-31', plots = True, prob_thr = '100'):
    """
    df - dataframe(pyspark) c предсказаниями
    words_list - list слов, по которым нужны предсказания
    precision_thr - порог точности, по которому хотим сделать отсечение, если отсекать не надо = 0
    plot - построить/не построить графики
    """
    
    prob_col = 'prob_{}'.format(prob_thr)
    target_col = 'target{}_new'.format(prob_thr)    
    
    df_words = df[(df['word'].isin(words_list))].toPandas()
    df_words[prob_col] = df_words[prob_col].astype(np.float64)
    df_words[target_col] = df_words[target_col].astype(np.int8)
    
    df_avg_max, df_anl_max, prs_max, rcl_max, thr_max = precision_recall(df_words, group_by = 'max', last_date_analysis = last_date_analysis, prob_thr = prob_thr)
    df_avg_mean, df_anl_mean, prs_mean, rcl_mean, thr_mean = precision_recall(df_words, group_by = 'mean', last_date_analysis = last_date_analysis, prob_thr = prob_thr)
        
    pr_rc_th_mean = pd.DataFrame({"pr": prs_mean, 'rcl': rcl_mean, 'thr': [0] + list(thr_mean)})
    pr_rc_th_max = pd.DataFrame({"pr": prs_max, 'rcl': rcl_max, 'thr': [0] + list(thr_max)})
    rec_mean =  pr_rc_th_mean[pr_rc_th_mean['pr'] >= precision_thr]['rcl'].values[0]
    rec_max =  pr_rc_th_max[pr_rc_th_max['pr'] >= precision_thr]['rcl'].values[0]
    
    print('recall mean = {}'.format(rec_mean))
    print('recall max = {}'.format(rec_max))
    
    if rec_mean < rec_max:
        thr_new = pr_rc_th_max[pr_rc_th_max['pr'] > precision_thr]['thr'].values[0]
        df_avg, df_anl = df_avg_max, df_anl_max
        pr_rc_th_new = pr_rc_th_max
    else:
        thr_new = pr_rc_th_mean[pr_rc_th_mean['pr'] > precision_thr]['thr'].values[0]
        df_avg, df_anl = df_avg_max, df_anl_max
        pr_rc_th_new = pr_rc_th_mean
    sample_by_new = df_avg[(df_avg['control_date'] == sample_date) & (df_avg[prob_col] > thr_new)]
    #for size of sample
    pr_rc_th_max['n'] = (pr_rc_th_max['rcl']/pr_rc_th_max['pr'])*df_anl_max[target_col].sum()
    pr_rc_th_mean['n'] = (pr_rc_th_mean['rcl']/pr_rc_th_mean['pr'])*df_anl_mean[target_col].sum()
    
    if plots:
        fig, axs = plt.subplots(1, 2, figsize=(50, 10))
        
        axs[0].set_title(u'precision-recall curve', fontsize=20)
        axs[0].plot(rcl_mean, prs_mean, label='target {} - {}'.format(prob_thr, 'mean'),  alpha=0.9, color = 'darkblue')
        axs[0].plot(rec_mean, precision_thr, marker = 'o', color = 'blue', markersize = 10.)
        axs[0].plot(rcl_max, prs_max, label='target {} - {}'.format(prob_thr, 'max'),  alpha=0.9, color = 'darkred')
        axs[0].plot(rec_max, precision_thr, marker = '*', color = 'red', markersize = 10.)
        
        axs[0].set_xlabel(u'recall',fontsize=16)
        axs[0].set_ylabel(u'precision',fontsize=16)
        axs[0].legend()
        axs[0].grid()
        
        axs[1].set_title(u'size of sample by precision', fontsize=20)
        axs[1].plot(pr_rc_th_max['n'], pr_rc_th_max['pr'], label = 'max', color = 'darkred')
        axs[1].plot(pr_rc_th_mean['n'], pr_rc_th_mean['pr'], label = 'mean', color = 'darkblue')
        axs[1].plot(pr_rc_th_max[pr_rc_th_max['pr'] >= precision_thr]['n'].values[0], precision_thr, marker = 'o', markersize = 10., color = 'red')
        axs[1].plot(pr_rc_th_mean[pr_rc_th_mean['pr'] >= precision_thr]['n'].values[0], precision_thr, marker = 'o', markersize = 10., color = 'blue')
        axs[1].set_yticks(np.arange(0, 1, 0.1))
        axs[1].set_xlabel(u'size',fontsize=16)
        axs[1].set_ylabel(u'precision',fontsize=16)
        axs[1].legend()
        axs[1].grid();
              
        
        
    return  sample_by_new, df_anl

In [ ]:
df = spark.read.table("prx_ml360_credit_products_b2b_custom_cib_ml360_credit_products.predictions_commertial_interest_dt")

In [ ]:
distinct_dates = df.select('control_date').distinct().collect()
analysis_date, predictions_date = sorted([dt.strptime(i[0], '%Y-%m-%d').date() for i in distinct_dates if len(distinct_dates) == 2])
predictions_date, analysis_date

In [ ]:
report_name = 'report_{}.xlsx'.format(name)

inns = spark.createDataFrame(pd.read_excel(report_name, 
                                          usecols=['ИНН'], 
                                          dtype={'ИНН': 'str','inn': 'str'},
                                          engine='openpyxl'
                                         ).rename(columns={'ИНН': 'inn'}))

In [ ]:
df_f = df.join(inns, 'inn', 'leftsemi')

In [ ]:
#порог точности
precision_threshold = 0.4

In [ ]:
#предсказания с точностью 80
#prob_thr = '100' - порог покупки или продажи: 100, 300, 500, 1m или 2m (строковый) - нужен, чтобы внутри функции выбрались нужные вероятности и таргет (колонки переименовывать не надо)

sample_by_new, df_anlisys  = create_sample(df_f, words_list, prob_thr = '100', precision_thr = precision_threshold, 
                 last_date_analysis = str(analysis_date), sample_date = str(predictions_date), plots = True)

In [ ]:
sample_by_new['inn'].nunique()

In [ ]:
comm_int_name = 'comm_int_{}.xlsx'.format(name)
#save
sample_by_new.to_excel(comm_int_name, index = False)

In [ ]:
rep = pd.read_excel(report_name, dtype={'ИНН': 'str','inn': 'str'}, engine='openpyxl').rename(columns={'inn': 'ИНН'})
comm_interest = pd.read_excel(comm_int_name, dtype={'inn': 'str'}, engine='openpyxl').rename(columns={'inn': 'ИНН'})

In [ ]:
rep.merge(comm_interest, on='ИНН', how='left')\
   .to_excel('final_report_{}.xlsx'.format(name), index=False)

# Сохранение

In [ ]:
import pandas as pd
from cryptography.fernet import Fernet

In [ ]:
df = pd.read_excel("Аналитика Силур финал2.xlsx", dtype={'trans_sum':int})

In [ ]:
df['revenue'] = df['revenue'].fillna(0)

In [ ]:
df['revenue'] = df['revenue'].astype(int)

In [ ]:
df['crm_holding_id'] = df['crm_holding_id'].fillna(0)

In [ ]:
df['crm_holding_id'] = df['crm_holding_id'].astype(int)

In [ ]:
def generate_key():  # функция для создания ключа для хэширования
    return Fernet.generate_key()

def encrypt_data(key, data):  # шифруем
    fernet = Fernet(key)
    encrypted = fernet.encrypt(data.encode())
    return encrypted

In [ ]:
key = generate_key()

In [ ]:
key

In [ ]:
# Шифруем данные и добавляем новые столбцы
df['hash_time'] = df['inn'].astype(str).apply(lambda x: encrypt_data(key, x))
df['hash_type'] = df['revenue'].astype(str).apply(lambda x: encrypt_data(key, x))
df['hash_hold'] = df['crm_holding_id'].astype(str).apply(lambda x: encrypt_data(key, x))

In [ ]:
a =df[['trans_sum', 'trans_cnt', 're_word', 'fa_word', 'type',
       'crm_segment', 'okfs_cd', 'active_flag', 'status_nm', 'okato', 'okato_cd', 'region', 'f_ocryg', 
       'okved', 'hash_type', 'year_revenue', 'hash_time', 'hash_hold', 'sum_group', 'cnt_group', 'z_score']]

In [ ]:
a.to_excel("Силур сводный отчет.xlsx", index=False)

# Поиск клиентов конкурентов, без слов

In [ ]:
words_finadv_ = sbersov_table
words_finadv_f = words_finadv_[
    (words_finadv_['short_dt'] >= '2024-08-01')\
   &(words_finadv_['short_dt'] <= '2025-08-01')
]

In [ ]:
inn_concur = []

concur_df = spark.createDataFrame(pd.DataFrame(inn_concur, columns=['inn']))

In [ ]:
concur_inn_dt = words_finadv_f.join(concur_df, on= words_finadv_f.inn_kt == concur_df.inn, how='leftsemi')\
                              .withColumnRenamed('inn_dt', 'inn')

In [ ]:
concur_inn_dt_grouped = concur_inn_dt.groupby('inn').agg(F.countDistinct('id').alias('trans_cnt'),
                                                         F.sum('c_sum').cast(FloatType()).alias('trans_sum'),
                                                         F.collect_set('word').alias('fa_word')\
                                                        )

In [ ]:
tpm_plat = "arnsdpsbx_team_monetization_prt_adhoc.dvo_none_b2b_pilot_russilika_concur"

In [ ]:
(concur_inn_dt_grouped
 .repartition(1)
.write
.mode('overwrite').format('parquet')
.option('compression','snappy')
 #.repartition(1)
 #.partiotionBy('month')
.saveAsTable(tpm_plat))

In [ ]:
concur_inn_dt_grouped = spark.table("arnsdpsbx_team_monetization_prt_adhoc.dvo_none_b2b_pilot_russilika_concur")

In [ ]:
concur_inn_dt_grouped.count()

In [ ]:
concur_inn_dt_filtered_1 = concur_inn_dt_grouped.join(crm_filtered, 'inn', 'inner')

In [ ]:
concur_inn_dt_filtered_1.count()

In [ ]:
concur_inn_dt_filtered_2 = concur_inn_dt_filtered_1.join(okfs, 'inn', 'inner')

In [ ]:
okfs_and_segment5 = concur_inn_dt_filtered_2.count()
okfs_and_segment5

In [ ]:
concur_inn_dt_filtered_3 = concur_inn_dt_filtered_2.join(inn_wth_regions, 'inn', 'inner')

In [ ]:
after_region5 = concur_inn_dt_filtered_3.count()
after_region5

In [ ]:
concur_inn_dt_filtered_4 = concur_inn_dt_filtered_3.join(okved_, 'inn', 'inner')

In [ ]:
after_okved5 = concur_inn_dt_filtered_4.count()
after_okved5

In [ ]:
concur_inn_dt_filtered_5 = concur_inn_dt_filtered_4.join(F.broadcast(already_clients), on = 'inn', how = 'leftanti')\
                                                   .join(F.broadcast(already_clients_1), on = 'inn', how = 'leftanti')
#                                                    .join(inn_clients_iskl, on = 'inn', how = 'leftanti')

In [ ]:
after_active5 = concur_inn_dt_filtered_5.count()
after_active5

In [ ]:
get_sum = lambda x: int(x.select(F.sum('trans_sum')).collect()[0][0])

In [ ]:
get_sum(concur_inn_dt_filtered_5)

In [ ]:
concur_inn_dt_filtered_6 = concur_inn_dt_filtered_5.join(revenue_filtered, on = 'inn', how = 'inner')

In [ ]:
concur_inn_dt_filtered_6.count()

In [ ]:
concur_inn_dt_filtered_7 = concur_inn_dt_filtered_6.filter((concur_inn_dt_filtered_6.trans_sum >= 300_000) & 
                                                           (concur_inn_dt_filtered_6.trans_cnt >= 1))

In [ ]:
concur_inn_dt_filtered_7.count()

In [ ]:
concur_inn_dt_filtered_7.limit(100).toPandas()

In [ ]:
tpm_plat13 = "arnsdpsbx_team_monetization_prt_adhoc.dvo_none_b2b_pilot_russilika__f_l_tmp_plat"

In [ ]:
(concur_inn_dt_filtered_7
 .repartition(1)
.write
.mode('overwrite').format('parquet')
.option('compression','snappy')
 #.repartition(1)
 #.partiotionBy('month')
.saveAsTable(tpm_plat13))

In [ ]:
concur_inn_dt_filtered_7.select(F.col("inn")).toPandas()